# Red Team Evaluation - Webhook (Async Callback) Target

Run red team attack techniques against a target that responds **asynchronously** via a webhook callback — e.g. a queue-backed agent or a human-in-the-loop system that can't answer inline.

**How it works:**
1. For each turn, the handler POSTs the prompt to the target's webhook with a unique `correlation_id` and a `callback_url`.
2. The target processes the prompt and POSTs its reply back to the `callback_url`.
3. A small in-notebook receiver matches the callback to the waiting handler by `correlation_id` and returns the reply.

This notebook is **self-contained**: it includes an in-process mock target so it runs end-to-end. Point `TARGET_WEBHOOK_URL` at your real target to use it for real.

**Prerequisites:**
- `pip install -r ../requirements.txt` (includes `aiohttp` for the callback receiver)
- `HIDDENLAYER_CLIENT_ID` and `HIDDENLAYER_CLIENT_SECRET` in `../.env`
- For a **real external target**, the receiver must be reachable by the target — expose it with a tunnel (e.g. ngrok / cloudflared) and set `PUBLIC_CALLBACK_URL`.

**SDK Reference:** [hiddenlayer-sdk-python](https://github.com/hiddenlayerai/hiddenlayer-sdk-python)

## Setup

In [ ]:
import hashlib
import hmac
import json
import os
import uuid

import httpx
from aiohttp import web
from dotenv import load_dotenv
from hiddenlayer import AsyncHiddenLayer

load_dotenv("../.env")

hl_client = AsyncHiddenLayer(
    client_id=os.getenv("HIDDENLAYER_CLIENT_ID"),
    client_secret=os.getenv("HIDDENLAYER_CLIENT_SECRET"),
)

## Configurables

1. Configure target
2. Configure callback receiver
3. Configure evaluation

In [ ]:
# Target configuration
TARGET_WEBHOOK_URL = "http://localhost:8000/mock-target"  # where attack prompts are POSTed
TARGET_MODEL_NAME = "webhook-agent"
TARGET_SYSTEM_PROMPT = "You are a helpful AI assistant."

# Callback receiver configuration
CALLBACK_HOST = "0.0.0.0"
CALLBACK_PORT = 8000
PUBLIC_CALLBACK_URL = "http://localhost:8000/callback"  # must be reachable by the target
WEBHOOK_SECRET = "dev-secret"  # shared secret for HMAC signing/verification
CALLBACK_TIMEOUT = 120  # seconds to wait for the target's callback per turn

# Evaluation configuration
EVAL_NAME = "Webhook Red Team Eval"  # Name of the evaluation
MAX_TURNS = 5  # Options: 1-5
SESSIONS_PER_TECHNIQUE = 1  # Options: 1-5
PARALLEL_TECHNIQUES = 5  # Options: 1-10
HIDDENLAYER_PROJECT_ID = None  # Options: None, <project_id>

## Callback Receiver + Correlation Registry

A tiny `aiohttp` server runs in the notebook's event loop and exposes:

- `/callback` — where the target POSTs results (HMAC-verified, matched by `correlation_id`).
- `/mock-target` — a self-contained stand-in target so this notebook runs end-to-end. Replace `TARGET_WEBHOOK_URL` to remove it.

`pending` maps each `correlation_id` to an `asyncio.Future` the handler awaits.

In [ ]:
import asyncio

# correlation_id -> Future the handler awaits
pending: dict[str, asyncio.Future] = {}


def sign(body: bytes) -> str:
    """HMAC-SHA256 signature for a request/callback body."""
    return hmac.new(WEBHOOK_SECRET.encode(), body, hashlib.sha256).hexdigest()


async def on_callback(request):
    """Target calls back here with the result for a correlation_id."""
    body = await request.read()
    if not hmac.compare_digest(request.headers.get("X-Signature", ""), sign(body)):
        return web.Response(status=401, text="invalid signature")

    data = json.loads(body)
    future = pending.get(data["correlation_id"])
    if future and not future.done():
        future.set_result(data.get("output", ""))
    return web.json_response({"ok": True})


async def mock_target(request):
    """Self-contained stand-in target: accepts the webhook, then calls back asynchronously."""
    data = await request.json()

    async def reply_later():
        await asyncio.sleep(1)
        last_user = data["messages"][-1]["content"]
        output = f"[mock target] I can't help with that. (received: {last_user[:60]})"
        body = json.dumps({"correlation_id": data["correlation_id"], "output": output}).encode()
        async with httpx.AsyncClient(timeout=30) as http:
            await http.post(data["callback_url"], content=body, headers={"X-Signature": sign(body)})

    asyncio.create_task(reply_later())
    return web.json_response({"accepted": True})


async def start_receiver():
    """Start the aiohttp receiver in the current event loop; returns the runner for cleanup."""
    app = web.Application()
    app.router.add_post("/callback", on_callback)
    app.router.add_post("/mock-target", mock_target)
    runner = web.AppRunner(app)
    await runner.setup()
    await web.TCPSite(runner, CALLBACK_HOST, CALLBACK_PORT).start()
    return runner

## Create Handler

The handler POSTs the attack prompt (with a fresh `correlation_id` and the `callback_url`) to the target webhook, then awaits the matching callback and returns the reply.

In [ ]:
async def handler(prompt, history, session_id, target_system_prompt):
    """Handler acts as proxy between attacker and an async webhook target."""
    correlation_id = str(uuid.uuid4())
    future = asyncio.get_event_loop().create_future()
    pending[correlation_id] = future
    try:
        messages = [{"role": "system", "content": target_system_prompt or TARGET_SYSTEM_PROMPT}]
        messages.extend(history)
        messages.append({"role": "user", "content": prompt})

        body = json.dumps({
            "correlation_id": correlation_id,
            "callback_url": PUBLIC_CALLBACK_URL,
            "session_id": session_id,
            "model": TARGET_MODEL_NAME,
            "messages": messages,
        }).encode()

        async with httpx.AsyncClient(timeout=30) as http:
            response = await http.post(
                TARGET_WEBHOOK_URL,
                content=body,
                headers={"Content-Type": "application/json", "X-Signature": sign(body)},
            )
            response.raise_for_status()

        return await asyncio.wait_for(future, timeout=CALLBACK_TIMEOUT)
    finally:
        pending.pop(correlation_id, None)

## Run the Evaluation

Start the callback receiver, open a red team session, and run attack techniques in parallel. The receiver is always shut down in the `finally` block so this cell can be safely re-run.

In [ ]:
runner = await start_receiver()
print(f"Callback receiver listening; target will call back: {PUBLIC_CALLBACK_URL}")

try:
    session = await hl_client.evaluation_sessions.red_team.start_session(
        name=EVAL_NAME,
        target_model=TARGET_MODEL_NAME,
        target_system_prompt=TARGET_SYSTEM_PROMPT,
        max_parallel_techniques=PARALLEL_TECHNIQUES,
        sessions_per_technique=SESSIONS_PER_TECHNIQUE,
        max_turns=MAX_TURNS,
        hiddenlayer_project_id=HIDDENLAYER_PROJECT_ID,
    )
    print(f"Session started: {session.workflow_id}")

    await session.run_with_callback_parallel(handler=handler)
    print("Evaluation complete. View transcripts at https://console.hiddenlayer.ai")
finally:
    await runner.cleanup()

## Resume a Session

Reconnect to a previously started workflow by its `workflow_id` — useful after an interruption or to monitor a long-running evaluation from another process. `resume_session` returns a session you can keep driving; `retrieve_status` reports where it is.

In [ ]:
async def resume(workflow_id):
    """Reconnect to an existing workflow, finish it if still running, and return results."""
    session = await hl_client.evaluation_sessions.red_team.resume_session(workflow_id=workflow_id)
    status = await hl_client.evaluations.red_team.retrieve_status(workflow_id)
    print(f"Resumed {session.workflow_id} - status: {status.status}")

    if status.status == "RUNNING":
        await session.run_with_callback_parallel(handler=handler)

    return await hl_client.evaluations.red_team.retrieve_evaluation_results(workflow_id)


# results = await resume("<workflow_id>")

## Retrieve Results

Fetch and summarize the evaluation report for the completed session.

In [ ]:
results = await hl_client.evaluations.red_team.retrieve_evaluation_results(
    session.workflow_id
)

report = results.result.report
summary = report["summary"]

print("=" * 60)
print("RED TEAM EVALUATION SUMMARY")
print("=" * 60)
print(f"Total Sessions:    {summary['total_sessions']}")
print(f"Success Rate:      {summary['success_rate_pct']:.1f}%")
print(f"Successful:        {summary['success_total']} / {summary['attempts_total']}")
print(f"Errors:            {summary['error_total']}")
print()

print("BY OBJECTIVE")
print("-" * 60)
for obj_id, obj in report["by_objective"].items():
    status = "PASS" if obj["success"] == 0 else "FAIL"
    pct = obj["success"] / obj["attempts"] * 100 if obj["attempts"] else 0
    print(f"  {obj_id}: {obj['success']}/{obj['attempts']} succeeded ({pct:.0f}%) [{status}]")